In [ ]:
# Notebooks live in notebooks/; make the repo root importable (ppo, train_ppo, ...).
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "ppo" / "paths.py").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))


In [ ]:
from pathlib import Path
from train_ppo import PPOTrainingConfig, PPOTrainer, run_sinr_evaluation, RewardWeights
import pandas as pd, matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D

from matplotlib.collections import LineCollection

from stable_baselines3 import PPO 
from train_ppo import FlyingBaseStationEnv
from stable_baselines3.common.monitor import Monitor
import numpy as np
import plotly.express as px
import pandas as pd

from pathlib import Path
from matplotlib.colors import ListedColormap


In [ ]:
def next_run_dir(root: Path, stem: str = "run"):
    root.mkdir(exist_ok=True)
    idx = 1
    while True:
        candidate = root / f"{stem}_{idx:03d}"
        if not candidate.exists():
            candidate.mkdir()
            return candidate
        idx += 1
        
def latest_run_path(root: Path, stem: str = "run"):
    candidates = sorted(root.glob(f"{stem}_*"))
    if not candidates:
        raise FileNotFoundError("No saved runs.")
    return candidates[-1]

def run_dir_path(root: Path, run_id: int, stem: str = "run") -> Path:
    path = root / f"{stem}_{run_id:03d}"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found")
    return path

def run_dir_by_name(root: Path, name: str) -> Path:
    path = root / name
    if not path.exists():
        raise FileNotFoundError(f"{path} not found")
    return path  

In [ ]:
# Paths come from ppo/paths.py -> secrets.env (git-ignored). Nothing
# machine-specific is written into this notebook.
from ppo.paths import REPO_ROOT, RUNS_DIR

repo_root = REPO_ROOT
runs_dir = RUNS_DIR
monitor_path = repo_root / "ppo_logs" / "monitor.csv"

max_episode_steps = 30

In [ ]:
custom_mbs = [(1000.0,1000.0), (3500.0,2200.0)]

config = PPOTrainingConfig(
    num_fbs = 2,
    total_timesteps = 3_000,  
    max_episode_steps = max_episode_steps,
    ent_coef = 0.7,
    action_scale = 0.9,
    learning_rate = 1e-4,
    reward_weights=RewardWeights(beta=0.8, gamma=0.0, fbs_weight=0.4, fbs_exponent=1.0),
    # Custom world dimensions and MBS location overrides
    world_width= 4000.0,
    world_height= 3000.0,
    num_mbs=len(custom_mbs),
    mbs_locations=custom_mbs
)
trainer = PPOTrainer(config)
try:
    trainer.train()
    run_dir = next_run_dir(repo_root / "ppo_runs")
    trainer.save(run_dir / "ppo_fbs_agent")
finally:
    trainer.close()


In [ ]:
df = pd.read_csv(monitor_path, comment="#")
df['rolling_reward'] = df['r'].rolling(window=10).mean()
df.plot(x='t', y='rolling_reward')

### Testing

In [ ]:
"""
latest run directory
"""
target_dir = latest_run_path(runs_dir)


"""
specific run directory
uncomment and set run_name to use a specific run
"""
run_name = "run_039"
target_dir = run_dir_by_name(runs_dir, run_name)

print(f"Loading model from: {target_dir}")

# custom_mbs = [(1000.0,1000.0), (3500.0,2200.0)]
max_episode_steps_test = 100
model = PPO.load(target_dir / "ppo_fbs_agent")
# model = PPO.load(target_dir / "ppo_fbs_agent_nfbs2") # for 2 FBS setting
env = FlyingBaseStationEnv(
    num_fbs = 1,
    max_episode_steps = max_episode_steps_test,
    action_scale = 0.6,
    # for non default environment settings 
    world_width=2000.0,
    world_height=1500.0,
    # num_mbs=len(custom_mbs),
    # mbs_locations=custom_mbs
)
# xs, ys = zip(*custom_mbs)
# mbs_x = list(xs)
# mbs_y = list(ys)
mbs_x = np.array(env.mbs_x).ravel()
mbs_y = np.array(env.mbs_y).ravel()

"""comment this block if randomizing the reset state"""
custom_state = np.array([800.0, 800.0, 100.0, 10.5, 1.0], dtype=np.float32) 
# custom_state = np.array(
#     [
#         [0.0, 0.0, 100.0, 7.0, 0.0],
#         [2000.0, 300.0, 80.0, 8.5, 0.0],
#     ],
#     dtype=np.float32,
# )
obs, _ = env.reset()
env._state = custom_state.reshape(-1).copy()
obs = env._get_obs()

"""use if evaluating with regular environment random state"""
# obs, _ = env.reset()

states = [obs.copy()]
rewards, infos = [], []
for step in range(max_episode_steps_test):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, term, trunc, info = env.step(action)
    states.append(obs.copy())
    rewards.append(reward)
    infos.append(info)
    if term or trunc:
        break


In [ ]:
# """
# latest run directory
# """
# target_dir = latest_run_path(runs_dir)


# """
# specific run directory
# uncomment and set run_name to use a specific run
# """
# # run_name = "run_000_cand3_ps_mark"
# run_name = "run_045"
# target_dir = run_dir_by_name(runs_dir, run_name)

# print(f"Loading model from: {target_dir}")
# # model_print = PPO.load(target_dir / "ppo_fbs_agent")
# model_print = PPO.load(target_dir / "ppo_fbs_agent_nfbs2") # for 2 FBS setting

# hyperparams = {
#     "learning_rate": model_print.learning_rate,
#     "n_steps": model_print.n_steps,
#     "batch_size": model_print.batch_size,
#     "gamma": model_print.gamma,
#     "gae_lambda": model_print.gae_lambda,
#     "clip_range": model_print.clip_range,
#     "ent_coef": model_print.ent_coef,
#     "vf_coef": model_print.vf_coef,
#     "max_grad_norm": model_print.max_grad_norm,
#     "policy_kwargs": model_print.policy_kwargs,
# }
# hyperparams

In [ ]:
states_df = pd.DataFrame(states)
states_df

In [ ]:
def flatten_states(state_history, num_fbs):
    arr = np.asarray(state_history, dtype=float)
    per_fbs_dim = 5 * num_fbs

    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    if arr.shape[1] < per_fbs_dim:
        raise ValueError(f"Expected at least {per_fbs_dim} entries per state, got {arr.shape[1]}")

    base = arr[:, :per_fbs_dim].reshape(-1, num_fbs, 5)
    cols = [
        (f"fbs{idx}", name)
        for idx in range(num_fbs)
        for name in ["x", "y", "height", "power", "power_status"]
    ]
    df = pd.DataFrame(base.reshape(len(base), -1), columns=pd.MultiIndex.from_tuples(cols))

    # Optional: capture the new global deltas if present
    if arr.shape[1] >= per_fbs_dim + 2:
        df[("global", "fbs_delta")] = arr[:, per_fbs_dim]
        df[("global", "total_delta")] = arr[:, per_fbs_dim + 1]

    return df

def plot_metrics(rewards, infos):
    df = pd.DataFrame(infos)
    df["reward"] = rewards
    fig, axes = plt.subplots(2, 2, figsize=(10,6), sharex=True)
    df["reward"].plot(ax=axes[0,0], title="Reward")
    df["total_connected"].plot(ax=axes[0,1], title="Connected Users")
    df["total_power"].plot(ax=axes[1,0], title="Total Power")
    df["avg_rate"].plot(ax=axes[1,1], title="Avg Rate (bps/Hz)")
    axes[1,0].set_xlabel("Step"); axes[1,1].set_xlabel("Step")
    plt.tight_layout()
    return df

def matlab_best_table_to_state_df(best_table, num_fbs=None):
    """
    best_table: array-like of shape (T, 5*num_fbs)
                columns are [x,y,z,power,status] per FBS, concatenated.
    num_fbs: optional. If None, inferred from column count.
    """
    arr = np.asarray(best_table, dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    if num_fbs is None:
        if arr.shape[1] % 5 != 0:
            raise ValueError(f"Expected columns multiple of 5, got {arr.shape[1]}")
        num_fbs = arr.shape[1] // 5

    expected = 5 * num_fbs
    if arr.shape[1] != expected:
        raise ValueError(f"Expected {expected} columns for num_fbs={num_fbs}, got {arr.shape[1]}")

    cols = []
    for idx in range(num_fbs):
        cols += [
            (f"fbs{idx}", "x"),
            (f"fbs{idx}", "y"),
            (f"fbs{idx}", "height"),
            (f"fbs{idx}", "power"),
            (f"fbs{idx}", "power_status"),
        ]

    state_df = pd.DataFrame(arr, columns=pd.MultiIndex.from_tuples(cols))
    return state_df

metrics_df = plot_metrics(rewards, infos)
state_df = flatten_states(states, env.num_fbs)

# fig, ax = plt.subplots(figsize=(6,5))
# ax.scatter(mbs_x, mbs_y, c="red", marker="x", s=80, label="MBS Locations")
# for i, (x, y) in enumerate(zip(mbs_x, mbs_y)):
#     ax.annotate(f"MBS {i}", (x, y), textcoords="offset points", xytext=(5,5))
# ax.legend()

# for fbs_idx in range(env.num_fbs):
#     ax.plot(state_df[(f"fbs{fbs_idx}", "x")], state_df[(f"fbs{fbs_idx}", "y")], label=f"FBS {fbs_idx}")
# ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_title("Trajectories")
# ax.grid(True)


"""advanced progression plotting FBS positions over time"""
def add_power_colored_path(ax, x, y, power, norm, cmap, linewidth=2.0, linestyle="--", alpha=0.95, zorder=4):
    """
    Draw a 2D path where each segment is colored by power.
    Uses a LineCollection so we avoid scatter clutter.
    """
    x = np.asarray(x); y = np.asarray(y); power = np.asarray(power)

    # Build line segments: shape (N-1, 2, 2)
    pts = np.column_stack([x, y]).reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)

    # Color each segment by the average power across its endpoints
    seg_power = 0.5 * (power[:-1] + power[1:])

    lc = LineCollection(
        segs,
        array=seg_power,
        cmap=cmap,
        norm=norm,
        linewidths=linewidth,
        alpha=alpha,
        zorder=zorder
    )
    lc.set_linestyle(linestyle)

    ax.add_collection(lc)
    return lc  # return handle (useful for colorbar)

def plot_fbs_xy_matlab_style(
    state_df,
    user_positions,              # (N,2) array
    mbs_x, mbs_y,                # scalar or array-like
    title="",
    xlim=None,
    ylim=None,
    user_alpha=0.06,
    user_size=8,
    user_color="0.75",
    min_gray=0.25,
    max_gray=0.98,
    legend_outside=True,
    extra_state_dfs=None,          # <-- add this
    run_labels=None,               # <-- add this (optional)
    path_linestyles=None           # <-- add this (optional)
):
    # --- detect how many fbss exist in df
    fbs_ids = sorted({lvl0 for (lvl0, lvl1) in state_df.columns if lvl0.startswith("fbs")})
    if not fbs_ids:
        raise ValueError("No FBS columns found (expected MultiIndex columns like ('fbs0','x')).")

    user_pos = np.asarray(user_positions)

    # --- global power normalization across all FBS ON points (one consistent colorbar)
    all_p_on = []
    for fbs in fbs_ids:
        s = state_df[(fbs, "power_status")].to_numpy().astype(int)
        p = state_df[(fbs, "power")].to_numpy()
        all_p_on.append(p[s == 1])
    all_p_on = np.concatenate([a for a in all_p_on if a.size > 0]) if any(a.size > 0 for a in all_p_on) else None

    if all_p_on is not None and all_p_on.size > 0:
        norm = mpl.colors.Normalize(vmin=float(all_p_on.min()), vmax=float(all_p_on.max()))
    else:
        # fallback if everything is OFF
        p_all = np.concatenate([state_df[(fbs, "power")].to_numpy() for fbs in fbs_ids])
        norm = mpl.colors.Normalize(vmin=float(p_all.min()), vmax=float(p_all.max()))

    base = plt.cm.Greys
    cmap = mpl.colors.LinearSegmentedColormap.from_list(
        "Greys_clipped",
        base(np.linspace(min_gray, max_gray, 256))
    )

    # --- IEEE-ish fonts (slightly smaller, as requested previously)
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "legend.fontsize": 8,
        "figure.dpi": 400
    })

    fig, ax = plt.subplots(figsize=(4.9, 4.2))
    runs = [state_df] + (extra_state_dfs or [])
    if run_labels is None:
        run_labels = [f"Run {i}" for i in range(len(runs))]
    if path_linestyles is None:
        path_linestyles = ["-"] * len(runs)
    # Users (faint background)
    ax.scatter(user_pos[:, 0], user_pos[:, 1],
               s=user_size, c=user_color, alpha=user_alpha, linewidths=0,
               label="_nolegend_")  # no legend entry

    # MBS
    mbs_x_arr = np.atleast_1d(mbs_x)
    mbs_y_arr = np.atleast_1d(mbs_y)
    ax.scatter(mbs_x_arr, mbs_y_arr, marker="x", s=90, label="MBS")
    for i, (mx, my) in enumerate(zip(mbs_x_arr, mbs_y_arr)):
        ax.annotate(f"MBS {i}", (mx, my), textcoords="offset points", xytext=(6, 6))

    # Per-FBS plotting (marker shapes differentiate fbss without color)
    marker_cycle = ["o", "s", "^", "D", "v", "P", "X"]  # enough for a few FBSs
    sc_for_cbar = None


    for r_idx, (df_run, run_name, ls) in enumerate(zip(runs, run_labels, path_linestyles)):
        fbs_ids = sorted({lvl0 for (lvl0, lvl1) in df_run.columns if lvl0.startswith("fbs")})

        for k, fbs in enumerate(fbs_ids):
            mk = marker_cycle[k % len(marker_cycle)]

            x = df_run[(fbs, "x")].to_numpy()
            y = df_run[(fbs, "y")].to_numpy()
            p = df_run[(fbs, "power")].to_numpy()
            s = df_run[(fbs, "power_status")].to_numpy().astype(int)


            if 'RL' in run_name:
                stride = 10  # try 5, 10, 20 depending on how long RL is
                idx = np.arange(len(x))
                keep = (idx % stride) == 0
                on = (s == 1) & keep
                off = (s == 0) & keep

            else:
                on = (s == 1)
                off = ~on

            # backbone path
            ax.plot(x, y, linewidth=1.2, alpha=0.85, linestyle=ls, label=f"{run_name} {fbs} path")
            
            # ON: power-coded
            if np.any(on):
                sc = ax.scatter(x[on], y[on],
                                c=p[on], cmap=cmap, norm=norm,
                                s=38, marker=mk,
                                edgecolors="black", linewidths=0.25, alpha=0.95, label = "_nolegend_")
                                # label=f"{fbs} ON")
                sc_for_cbar = sc  # just keep last; all share same norm/cmap

            # OFF: hollow
            if np.any(off):
                ax.scatter(x[off], y[off],
                        s=38, marker=mk,
                        facecolors="none", edgecolors="black", linewidths=0.9,label = "_nolegend_")
                        # label=f"{fbs} OFF")

            # start/end markers (keep subtle to avoid clutter)
            ax.scatter(x[0], y[0], marker=mk, s=55, facecolors="none", edgecolors="black", linewidths=1.0)
            ax.scatter(x[-1], y[-1], marker="*", s=85, edgecolors="black", linewidths=0.4)


    # axes formatting
    ax.set_xlabel("x-coordinate (m)")
    ax.set_ylabel("y-coordinate (m)")
    if title:
        ax.set_title(title)
    ax.grid(alpha=0.25)

    if xlim is not None: ax.set_xlim(xlim)
    if ylim is not None: ax.set_ylim(ylim)

    # shared colorbar
    if sc_for_cbar is not None:
        cbar = fig.colorbar(sc_for_cbar, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Transmit Power")


    # legend outside (right)
    if legend_outside:
        status_handles = [
            Line2D([0], [0],
                marker='o', linestyle='None',
                markerfacecolor='0.4', markeredgecolor='black',
                markersize=6, label='ON (filled)'),
            Line2D([0], [0],
                marker='o', linestyle='None',
                markerfacecolor='none', markeredgecolor='black',
                markersize=6, label='OFF (hollow)')
        ]
        handles, labels = ax.get_legend_handles_labels()
        handles = handles + status_handles
        labels  = labels  + [h.get_label() for h in status_handles]
        ax.legend(
            handles, labels,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.18),  # move legend below the axes
            ncol=4,                       # number of items per row (adjust!)
            frameon=False,
            fontsize=9,
            handlelength=2.2,
            columnspacing=1.2
        )

        # ax.legend(
        # loc="center left",
        # bbox_to_anchor=(1.25, 0.5),   # push legend outside to the right
        # fontsize=7,
        # frameon=False
        # )
    else:
        ax.legend(loc="best", frameon=False)
        plt.tight_layout()
        # ax.legend(
        # loc="center left",
        # bbox_to_anchor=(1.25, 0.5),   # push legend outside to the right
        # frameon=False
        # )

    return fig, ax

def plot_fbs_xy_progression_with_context(
    state_df,
    metrics_df,
    mbs_x, mbs_y,
    fbs_id="fbs0",
    title="FBS 2D Trajectory Relative to MBS (Power + ON/OFF)",
    xlim=None,
    ylim=None,
    user_positions_step_index=0,    # which snapshot to use from metrics_df["user_positions"]
    user_alpha=0.1,                # faint visibility
    user_size=8,                    # small points
    user_color="0.75",              # light gray
    min_gray=0.25,                  # prevents power colormap from going to pure white
    max_gray=0.98,                  # keep top end near black-ish
):
    """
    - User positions plotted faintly (background context)
    - FBS path and states overlayed
    - ON points colored by power with a grayscale range that stays visible at low power
    - OFF points hollow
    - xlim/ylim supported for consistent plotting across scenarios
    """

    # -----------------------------
    # Extract FBS series (MultiIndex)
    # -----------------------------
    x = state_df[(fbs_id, "x")].to_numpy()
    y = state_df[(fbs_id, "y")].to_numpy()
    p = state_df[(fbs_id, "power")].to_numpy()
    s = state_df[(fbs_id, "power_status")].to_numpy().astype(int)

    on_mask = (s == 1)
    off_mask = ~on_mask

    # -----------------------------
    # User positions (background)
    # metrics_df["user_positions"][k] => (N,2)
    # -----------------------------
    user_pos = metrics_df["user_positions"].iloc[user_positions_step_index]
    user_pos = np.asarray(user_pos)

    # -----------------------------
    # Power normalization (ON only)
    # -----------------------------
    if np.any(on_mask):
        p_on = p[on_mask]
        norm = mpl.colors.Normalize(vmin=np.min(p_on), vmax=np.max(p_on))
    else:
        norm = mpl.colors.Normalize(vmin=np.min(p), vmax=np.max(p))

    # -----------------------------
    # Colormap that never becomes "white"
    # (so low power still shows on white background)
    # -----------------------------
    base = plt.cm.Greys
    cmap = mpl.colors.LinearSegmentedColormap.from_list(
        "Greys_clipped",
        base(np.linspace(min_gray, max_gray, 256))
    )

    # -----------------------------
    # IEEE-ish matplotlib style
    # -----------------------------
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 9,          # ↓ overall text
        "axes.labelsize": 9,     # ↓ axis labels
        "legend.fontsize": 8,    # ↓ legend
        "axes.titlesize": 9,     # optional but recommended
        "figure.dpi": 300
    })

    fig, ax = plt.subplots(figsize=(4.9, 4.2))

    # -----------------------------
    # 1) Users (faint background)
    # -----------------------------
    ax.scatter(
        user_pos[:, 0], user_pos[:, 1],
        s=user_size,
        c=user_color,
        alpha=user_alpha,
        linewidths=0,
        label="Users (snapshot)"
    )

    # -----------------------------
    # 2) MBS location(s)
    # -----------------------------
    mbs_x_arr = np.atleast_1d(mbs_x)
    mbs_y_arr = np.atleast_1d(mbs_y)

    ax.scatter(mbs_x_arr, mbs_y_arr, marker="x", s=90, label="MBS")
    for i, (mx, my) in enumerate(zip(mbs_x_arr, mbs_y_arr)):
        ax.annotate(f"MBS {i}", (mx, my), textcoords="offset points", xytext=(6, 6))

    # -----------------------------
    # 3) FBS backbone trajectory (thin line)
    # -----------------------------
    ax.plot(x, y, linewidth=1.2, alpha=0.85, label="FBS path")

    # -----------------------------
    # 4) ON states: power-coded
    # Add subtle black edge so low-power points still visible
    # -----------------------------
    sc = None
    if np.any(on_mask):
        sc = ax.scatter(
            x[on_mask], y[on_mask],
            c=p[on_mask],
            cmap=cmap,
            norm=norm,
            s=38,
            marker="o",
            edgecolors="black",      # key change: keeps low power visible
            linewidths=0.25,
            alpha=0.95,
            label="FBS ON (power-coded)"
        )

    # -----------------------------
    # 5) OFF states: hollow circles
    # -----------------------------
    if np.any(off_mask):
        ax.scatter(
            x[off_mask], y[off_mask],
            s=38,
            marker="o",
            facecolors="none",
            edgecolors="black",
            linewidths=0.9,
            label="FBS OFF"
        )

    # -----------------------------
    # 6) Start / End markers
    # -----------------------------
    ax.scatter(
        x[0], y[0],
        marker="s", s=60,
        facecolors="none", edgecolors="black",
        linewidths=1.2,
        label="Start"
    )
    ax.scatter(
        x[-1], y[-1],
        marker="*", s=95,
        edgecolors="black",
        linewidths=0.4,
        label="End"
    )

    # -----------------------------
    # Axes formatting + limits
    # -----------------------------
    ax.set_xlabel("x-coordinate (m)")
    ax.set_ylabel("y-coordinate (m)")
    ax.set_title(title)
    ax.grid(alpha=0.25)

    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    # -----------------------------
    # Colorbar (power)
    # -----------------------------
    if sc is not None:
        cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Transmit Power")

    # -----------------------------
    # Legend (clean / no frame)
    # -----------------------------
    # ax.legend(loc="best", frameon=False)
    ax.legend(
    loc="center left",
    bbox_to_anchor=(1.25, 0.5),   # push legend outside to the right
    frameon=False
    )

    # plt.tight_layout()
    # plt.tight_layout(rect=[0, 0, 0.82, 1])
    return fig, ax


best_table_matlab = pd.read_csv('ga_run_112.csv')
state_df_ga = matlab_best_table_to_state_df(best_table_matlab)  # auto infers num_fbs
user_pos = metrics_df["user_positions"].iloc[0]  # same snapshot you used earlier


fig, ax = plot_fbs_xy_matlab_style(
    state_df=state_df_ga,
    extra_state_dfs=[state_df],                # overlay RL on top
    run_labels=["GA (best/gen)", "RL (policy)"],
    path_linestyles=["-", "--"],               # differentiate paths cleanly
    user_positions=user_pos,
    mbs_x=mbs_x[:], mbs_y=mbs_y[:],
    xlim=(-100, 2100),
    ylim=(-100, 1600),
title="2D Trajectories of GA (best/gen) vs. RL Policy"
)


# fig, ax = plot_fbs_xy_matlab_style(
#     state_df=state_df,
#     extra_state_dfs=[state_df],                # overlay RL on top
#     run_labels=["GA (best/gen)", "RL (policy)"],
#     path_linestyles=["-", "--"],               # differentiate paths cleanly
#     user_positions=user_pos,
#     mbs_x=mbs_x[:], mbs_y=mbs_y[:],
#     xlim=(-100, 4100),
#     ylim=(-100, 3100),
# title="2D Trajectories of GA (best/gen) vs. RL Policy"
# )


plt.show()

In [ ]:
plt.rcParams.update({
    "font.size": 10,
    "font.family": "serif",
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "legend.fontsize": 9,
    "figure.dpi": 300
})

"""connectivity plot"""
# fig, ax = plt.subplots(figsize=(4.5, 3))
# ax.plot(metrics_df.index, metrics_df["total_connected"])
# ax.set_xlabel("Training Episode")
# ax.set_ylabel("Total Connected Users")
# ax.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()

"""fbs vs mbs connected users plot"""
# fig, ax = plt.subplots(figsize=(4.5, 3))
# ax.stackplot(
#     metrics_df.index,
#     metrics_df["mbs_connected"],
#     metrics_df["fbs_connected"],
#     labels=["MBS", "FBS"],
#     alpha=0.85
# )
# ax.set_xlabel("Training Episode")
# ax.set_ylabel("Connected Users")
# ax.legend(loc="upper left")
# ax.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()

# fig, ax = plt.subplots(figsize=(4.5, 3))
# ax.scatter(
#     metrics_df["total_power"],
#     metrics_df["total_connected"],
#     alpha=0.6
# )
# ax.set_xlabel("Total Transmitted Power")
# ax.set_ylabel("Total Connected Users")
# ax.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()


# fig, ax = plt.subplots(figsize=(4.5, 3))
# ax.plot(metrics_df.index, metrics_df["total_delta"], label="Total Δ Connectivity")
# ax.plot(metrics_df.index, metrics_df["fbs_delta"], linestyle="--", label="FBS Δ")
# ax.set_xlabel("Training Episode")
# ax.set_ylabel("Marginal Connectivity Gain")
# ax.legend()
# ax.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
fbs = "fbs0"
steps = state_df.index
height = state_df[(fbs, "height")]
x = state_df[(fbs, "x")]
y = state_df[(fbs, "y")]
h = state_df[(fbs, "height")]
p = state_df[(fbs, "power")]
ps = state_df[(fbs, "power_status")]
fbs_delta   = state_df[("global", "fbs_delta")]
total_delta = state_df[("global", "total_delta")]

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "figure.dpi": 300
})

"""power status plot"""
# fig, ax = plt.subplots(figsize=(4.5, 2.5))
# ax.step(state_df.index, ps, where="post", linewidth=1.5)
# ax.set_xlabel("Training Episode")
# ax.set_ylabel("FBS Active Status")
# ax.set_yticks([0, 1])
# ax.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()


# """power vs height plot"""
# fig, ax1 = plt.subplots(figsize=(4.8, 3.2))
# # --- Left axis: Height (solid line)
# ax1.plot(
#     steps,
#     height,
#     linewidth=1.6,
#     linestyle="-",
#     label="FBS Altitude"
# )

# ax1.set_xlabel("Training Episode")
# ax1.set_ylabel("FBS Altitude (m)")
# ax1.grid(alpha=0.3)
# # --- Right axis: Power (dotted line)
# ax2 = ax1.twinx()
# ax2.plot(
#     steps,
#     p,
#     linewidth=1.6,
#     linestyle=":",
#     label="Transmit Power"
# )
# ax2.set_ylabel("Transmit Power")
# # --- Combined legend (IEEE-style)
# lines_1, labels_1 = ax1.get_legend_handles_labels()
# lines_2, labels_2 = ax2.get_legend_handles_labels()
# ax1.legend(
#     lines_1 + lines_2,
#     labels_1 + labels_2,
#     loc="best",
#     frameon=False
# )
# plt.tight_layout()
# plt.show()

In [ ]:
# mbs_df = pd.DataFrame({"x": mbs_x, "y": mbs_y, "type": "MBS"})
# states_df = states.copy()
# states_df["type"] = states_df["power_status"].map({0: "FBS Off", 1: "FBS On"})

# fig_states = px.scatter(
#     states_df,
#     x="x",
#     y="y",
#     color="type",
#     symbol="type",
#     title="FBS Trajectory (Plotly)",
#     labels={"x": "X Position (m)", "y": "Y Position (m)"},
# )
# fig_mbs = px.scatter(
#     mbs_df,
#     x="x",
#     y="y",
#     color="type",
#     symbol="type",
# )

# for trace in fig_mbs.data:
#     fig_states.add_trace(trace)

# fig_states.update_layout(
#     xaxis=dict(range=[0, env.W]),
#     yaxis=dict(range=[0, env.H]),
#     legend=dict(title=""),
# )

In [ ]:
env.close()

## only use if evaluating single step

In [ ]:
target_dir = latest_run_path(runs_dir)
# run_name = "run_045"
# target_dir = run_dir_by_name(runs_dir, run_name)
print(f"Loading model from: {target_dir}")

model = PPO.load(target_dir / "ppo_fbs_agent")
# model = PPO.load(target_dir / "ppo_fbs_agent_nfbs2")
env = FlyingBaseStationEnv(
    num_fbs = 1,
    max_episode_steps = 1,
    # for non default environment settings 
    world_width=4000.0,
    world_height=3000.0,
    num_mbs=len(custom_mbs),
    mbs_locations=custom_mbs
)

	
# state = np.array([318.79, 0.0, 38.87, 7.0, 1.0], dtype=np.float32)
# state = state_df.iloc[-1].values
per_fbs_dim = 5 * env.num_fbs
state = state_df.iloc[-1].values[:per_fbs_dim]
state = state.reshape(env.num_fbs, 5)

total_connected, total_power, avg_rate, fbs_conn, mbs_conn, user_positions = run_sinr_evaluation(
    env._eng,
    env.antenna_fbs,
    env.antenna_mbs,
    env.mbs_cache,
    state[:, :4],     # all FBS tx params
    state[:, 4],      # all FBS power_status
    env.area_bounds,
    env.num_users,
    env.sinr_threshold,
    env.contains_mbs,
    env.mbs_x,
    env.mbs_y,
    env.mbs_height,
    env.mbs_power,
    num_fbs=env.num_fbs,
)
print(env.mbs_x, env.mbs_y)

weights = env.reward_weights
# reward = (
#     weights.beta_connected * total_connected
#     - weights.gamma_power * total_power
#     + weights.delta_avg_rate * avg_rate
# )
info = {
    "total_connected": total_connected,
    "total_power": total_power,
    "avg_rate": avg_rate,
    "fbs_connected": fbs_conn,
    "mbs_connected": mbs_conn,
    "user_positions": user_positions
}

# print({"total_connected": total_connected, "total_power": total_power, "avg_rate": avg_rate})
print(info)
env.close()